# Experiment 4 - Fully Connected Deep Neural Network for Multi-Class Classification

**Aim:** Build a fully connected deep neural network with two hidden layers
and classify a three-class, two-dimensional dataset.

**Theory in brief:** A deep feed-forward network stacks an input layer, two or
more hidden Dense layers, and an output layer. Each hidden layer applies a
nonlinear activation (`relu` here), so the network can learn hierarchical
representations. For three classes, the output layer has three neurons with
`softmax`, which turns the logits into a probability distribution. The model
is trained with `sparse_categorical_crossentropy` against integer labels and
optimized with Adam.

**Dataset:** Synthetic three-class Gaussian blobs generated in code. The three
centres are `(-2, 0)`, `(2, 0)`, and `(0, 2.5)`, each with 300 points and
standard deviation 0.8. No local file or download is needed.

**Expected result (original lab record):** about **96.67% test accuracy** with
a loss near **0.07** after 200 epochs.

**Runtime:** about 20-40 seconds on CPU.

## 1. Environment setup

In [ ]:
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ["PYTHONHASHSEED"] = "42"

import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

SEED = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")

## 2. Generate three Gaussian blobs

Each class is a cloud around a different centre. The clouds are close enough
that the classes overlap slightly, so the problem is not trivially separable.

In [ ]:
N_PER_CLASS = 300
CENTERS = np.array(
    [
        [-2.0, 0.0],
        [2.0, 0.0],
        [0.0, 2.5],
    ],
    dtype=np.float32,
)
CLASS_NAMES = ["blob_0", "blob_1", "blob_2"]

rng = np.random.default_rng(SEED)
X = np.vstack(
    [
        rng.normal(loc=centre, scale=0.8, size=(N_PER_CLASS, 2))
        for centre in CENTERS
    ]
).astype(np.float32)
y = np.repeat(np.arange(len(CENTERS), dtype=np.int64), N_PER_CLASS)

shuffle_indices = rng.permutation(len(X))
X = X[shuffle_indices]
y = y[shuffle_indices]

print("Feature matrix:", X.shape)
print("Labels        :", y.shape)
print("Class counts  :", np.bincount(y).tolist())

assert X.shape == (3 * N_PER_CLASS, 2)
assert y.shape == (3 * N_PER_CLASS,)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
colors = ["tab:red", "tab:green", "tab:blue"]
for class_index, class_name in enumerate(CLASS_NAMES):
    mask = y == class_index
    ax.scatter(
        X[mask, 0],
        X[mask, 1],
        s=18,
        color=colors[class_index],
        alpha=0.7,
        edgecolor="k",
        linewidth=0.3,
        label=class_name,
    )
ax.set_xlabel("x0")
ax.set_ylabel("x1")
ax.set_title("Three Gaussian blobs")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp04_blobs.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Train / validation / test split

The split is stratified so every subset contains all three classes. The model
is selected on the validation split, and the held-out test split is used only
for the final evaluation.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y,
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

print(f"Train      : {X_train.shape}, labels {np.bincount(y_train).tolist()}")
print(f"Validation : {X_val.shape}, labels {np.bincount(y_val).tolist()}")
print(f"Test       : {X_test.shape}, labels {np.bincount(y_test).tolist()}")

assert set(np.unique(y_train)) == {0, 1, 2}

## 4. Model: 2 -> 16 -> 16 -> 3

The network has two `relu` hidden layers of 16 units each and a three-way
`softmax` output. It is compiled with Adam (learning rate 0.01) and sparse
categorical cross-entropy.

In [ ]:
def build_deep_ffn(seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(2,), name="features"),
            tf.keras.layers.Dense(16, activation="relu", name="hidden_1"),
            tf.keras.layers.Dense(16, activation="relu", name="hidden_2"),
            tf.keras.layers.Dense(3, activation="softmax", name="class_probabilities"),
        ],
        name="deep_ffn_3class",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


model = build_deep_ffn()
model.summary()

## 5. Training for 200 epochs

In [ ]:
start_time = time.perf_counter()
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    shuffle=True,
    verbose=0,
)
training_seconds = time.perf_counter() - start_time

print(f"Training finished in {training_seconds:.2f} seconds")
print(f"Final training accuracy  : {history.history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")

## 6. Test evaluation

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
test_probabilities = model.predict(X_test, verbose=0)
test_predictions = test_probabilities.argmax(axis=1)

print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print()
print("Classification report:")
print(
    classification_report(
        y_test,
        test_predictions,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
)

In [ ]:
cm = confusion_matrix(y_test, test_predictions, labels=[0, 1, 2])

fig, ax = plt.subplots(figsize=(5.5, 4.5))
image = ax.imshow(cm, cmap="Blues")
ax.set_title("Test confusion matrix")
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(CLASS_NAMES)
ax.set_yticklabels(CLASS_NAMES)

for row in range(cm.shape[0]):
    for column in range(cm.shape[1]):
        color = "white" if cm[row, column] > cm.max() / 2 else "black"
        ax.text(column, row, str(cm[row, column]), ha="center", va="center", color=color)

fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp04_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history.history["loss"], label="Train", linewidth=2)
axes[0].plot(history.history["val_loss"], label="Validation", linewidth=2)
axes[0].set_title("Sparse categorical cross-entropy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="Train", linewidth=2)
axes[1].plot(history.history["val_accuracy"], label="Validation", linewidth=2)
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.0, 1.05)
axes[1].grid(alpha=0.3)
axes[1].legend()

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp04_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Learned decision regions

In [ ]:
x_min = X[:, 0].min() - 0.5
x_max = X[:, 0].max() + 0.5
y_min = X[:, 1].min() - 0.5
y_max = X[:, 1].max() + 0.5

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300),
)
grid = np.c_[xx.ravel(), yy.ravel()].astype(np.float32)
grid_probabilities = model.predict(grid, verbose=0)
grid_predictions = grid_probabilities.argmax(axis=1).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7.5, 6))
ax.contourf(
    xx,
    yy,
    grid_predictions,
    levels=[-0.5, 0.5, 1.5, 2.5],
    colors=["#f4b6b6", "#b6e2b6", "#b6c8f4"],
    alpha=0.8,
)
ax.contour(xx, yy, grid_predictions, levels=[0.5, 1.5], colors="k", linewidths=1.0)
for class_index, class_name in enumerate(CLASS_NAMES):
    mask = y_test == class_index
    ax.scatter(
        X_test[mask, 0],
        X_test[mask, 1],
        s=24,
        color=colors[class_index],
        edgecolor="k",
        linewidth=0.4,
        label=f"{class_name} (test)",
    )
ax.set_xlabel("x0")
ax.set_ylabel("x1")
ax.set_title("Deep FFN decision regions")
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "exp04_decision_regions.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Inference

In [ ]:
def predict_blob(x0, x1):
    point = np.array([[x0, x1]], dtype=np.float32)
    probabilities = model.predict(point, verbose=0)[0]
    class_index = int(np.argmax(probabilities))
    return CLASS_NAMES[class_index], float(probabilities[class_index]), probabilities


for x0, x1 in [(-2.0, 0.0), (2.0, 0.0), (0.0, 2.5), (0.0, 0.0)]:
    label, confidence, probabilities = predict_blob(x0, x1)
    print(f"Point ({x0:+.1f}, {x1:+.1f}) -> {label} (confidence {confidence:.4f})")
    print("  probabilities:", dict(zip(CLASS_NAMES, probabilities.round(4).tolist())))

print()
print("Example test rows:")
sample_indices = rng.choice(len(X_test), size=6, replace=False)
for sample_index in sample_indices:
    sample = X_test[sample_index]
    true_label = CLASS_NAMES[int(y_test[sample_index])]
    predicted_label, confidence, _ = predict_blob(float(sample[0]), float(sample[1]))
    print(
        f"  true={true_label:8s} predicted={predicted_label:8s} "
        f"confidence={confidence:.4f} features=({sample[0]:+.2f}, {sample[1]:+.2f})"
    )

## 10. Save model and metrics

In [ ]:
model_path = OUTPUT_DIR / "exp04_deep_ffn.keras"
model.save(model_path)

metrics = pd.DataFrame(
    [
        {
            "Test loss": float(test_loss),
            "Test accuracy": float(test_accuracy),
            "Training seconds": float(training_seconds),
        }
    ]
)
metrics.to_csv(OUTPUT_DIR / "exp04_metrics.csv", index=False)

print(f"Saved model to: {model_path.resolve()}")
print(metrics.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

reloaded = tf.keras.models.load_model(model_path)
reloaded_predictions = reloaded.predict(X_test, verbose=0).argmax(axis=1)
assert np.array_equal(reloaded_predictions, test_predictions)
print("Reload check passed: predictions are identical.")

## 11. Conclusion

The 2-16-16-3 network learns the three-blob problem and reaches high test
accuracy. The two `relu` hidden layers allow the model to form curved decision
regions rather than a single straight boundary. The softmax output gives a
calibrated class distribution, and sparse categorical cross-entropy handles
the integer labels directly.

The original lab sheet reports approximately **96.67% accuracy** and a loss
near **0.07**. The exact numbers in a rerun depend on initialization and the
train/validation/test split.

**Artifacts created at runtime (not committed):**

- `outputs/exp04_deep_ffn.keras`
- `outputs/exp04_metrics.csv`
- `outputs/exp04_blobs.png`
- `outputs/exp04_confusion_matrix.png`
- `outputs/exp04_training_curves.png`
- `outputs/exp04_decision_regions.png`

**Execution note:** the notebook was syntax-checked during repository
preparation; the full training run was not performed locally.